# Validation: gains and bandpass mismatch

One data set, three cases, with the **same noise realization** (1/f, not
correlated between detectors):

- **A**: no gain variation, no bandpass mismatch: the maps alone (the noise
  level);
- **B**: the same data with drifting gains (1% offsets, 3% random walk):
  gains ignored, or calibrated;
- **C**: gains and bandpass mismatch (bands shifted by +1.5, −1.0, +0.8,
  −1.3 GHz, dust with a spectral index varying across the sky): both ignored,
  calibration only, calibration and bandpass templates.

4 detectors in two orthogonal pairs, 6.7 hours (the orbital dipole turns
once), CMB and dust (3 mK on a Galactic band, structure at 1°, 10% polarized)
simulated at nside 128, maps at nside 16. All the solutions use the sky model
as alm (see the sky model notebook), so the effects of gains and bands are
isolated from those of the pixelization. The errors are the rms of map minus
the noiseless sky at the reference frequency (binned), out of and on the
Galactic band.

In [1]:
import dataclasses
import ducc0
import healpy as hp
import numpy as np
from pysanepic import CalibrationParameters, DetectorData, GLSParameters, OneOverFNoise, Pointing, make_maps

nside, fs, n = 16, 10.0, 240_000
hi, lmax = 128, 256
rng = np.random.default_rng(3)
npix, nhi = 12 * nside**2, 12 * hi**2
z, zh = hp.pix2vec(nside, np.arange(npix))[2], hp.pix2vec(hi, np.arange(nhi))[2]
cmb_alm = hp.map2alm(hp.smoothing(rng.normal(0, 1e-4, (3, nhi)), fwhm=np.radians(6)), lmax=lmax)
# dust: amplitude A (3 mK in a band, structure at 1 degree), 10% polarized, spectral index beta(p)
A = np.where(np.abs(zh) < 0.2, 3e-3, 2e-5) * (1 + 0.3 * hp.smoothing(rng.normal(0, 3, nhi), fwhm=np.radians(1)))
A = np.maximum(A, 0)
pa = hp.smoothing(rng.normal(0, 1, nhi), fwhm=np.radians(3)) * np.pi
dust = np.array([A, 0.1 * A * np.cos(2 * pa), 0.1 * A * np.sin(2 * pa)])
beta = 1.5 + 0.2 * hp.smoothing(rng.normal(0, 3, nhi), fwhm=np.radians(5))
nu0 = 140.0
shift = np.array([1.5, -1.0, 0.8, -1.3])  # GHz: band centre of each detector minus nu0
alm = lambda m: hp.map2alm(m, lmax=lmax)
F, F1 = alm(dust), alm(dust * beta / nu0)  # the dust at nu0 and its derivative dF/dnu
noise = OneOverFNoise(net_ukrts=35.0, fknee_hz=0.05, alpha=1.0, fmin_hz=1e-4)
t = np.arange(n)
lam = 2 * np.pi * t / n + rng.uniform(0, 2 * np.pi)
spin = 2 * np.pi * t / 3000.0
a = np.array([np.cos(lam), np.sin(lam), np.zeros(n)])
e2 = np.cross(a.T, [0, 0, 1]).T
vec = np.cos(1.3) * a + np.sin(1.3) * (np.cos(spin) * np.array([[0], [0], [1]]) + np.sin(spin) * e2)
theta, phi = np.arccos(np.clip(vec[2], -1, 1)), np.arctan2(vec[1], vec[0]) % (2 * np.pi)
psi = (spin + lam) % (2 * np.pi) - np.pi
solar = 3.36e-3 * np.array([0.3, -0.5, 0.8]) / np.linalg.norm([0.3, -0.5, 0.8])
dipole = solar @ vec + np.sum(2.7e-4 * np.array([-np.sin(lam), np.cos(lam), 0 * t]) * vec, axis=0)
loc = np.stack([theta, phi], axis=1)


def interp(alm, ang):
    i = ducc0.sht.synthesis_general(alm=alm[:1], spin=0, lmax=lmax, loc=loc, epsilon=1e-10)[0]
    q, u = ducc0.sht.synthesis_general(alm=alm[1:], spin=2, lmax=lmax, loc=loc, epsilon=1e-10)
    return i + q * np.cos(2 * ang) + u * np.sin(2 * ang)


def draw(n):
    return np.fft.irfft(np.fft.rfft(rng.normal(size=n)) / np.sqrt(noise.weights(n, fs)), n=n)

noise_tod = [draw(n) for _ in range(4)]  # the same noise in every case
gains = [1 + 0.01 * rng.normal() + 0.03 * np.cumsum(rng.normal(size=n)) / np.sqrt(n) for _ in range(4)]
angles = np.radians([0, 90, 45, 135])
cmb_t = [interp(cmb_alm, psi + a) for a in angles]
dust0 = [interp(F, psi + a) for a in angles]  # the dust at nu0
dust_b = [interp(alm(dust * ((nu0 + shift[i]) / nu0) ** beta), psi + a) for i, a in enumerate(angles)]  # through each band
pointing = Pointing(theta=theta, phi=phi, psi=psi, coordinates="E")


def make(gain, band):
    """The TOD of the 4 detectors, with or without gain variation and bandpass mismatch."""
    return [DetectorData(tod=(gains[i] if gain else 1.0) * (cmb_t[i] + (dust_b[i] if band else dust0[i]) + dipole) + noise_tod[i],
                         pointing=pointing, sampling_rate_hz=fs, noise=noise, pol_angle_rad=a, dipole=dipole, name=f"d{i}")
            for i, a in enumerate(angles)]


params = GLSParameters(chunk_s=2400.0, tol=1e-12)
truth = make_maps([DetectorData(tod=cmb_t[i] + dust0[i], pointing=pointing, sampling_rate_hz=fs,
                                noise=OneOverFNoise(net_ukrts=35.0), pol_angle_rad=a) for i, a in enumerate(angles)],
                  nside, coordinates="E", params=params)  # binned noiseless sky at nu0
model = cmb_alm + F
cal = CalibrationParameters(gain_period_s=1200.0)
ignore_gains = lambda data: [dataclasses.replace(d, tod=d.tod - dipole, dipole=None) for d in data]  # g = 1


def report(name, r):
    line = f"{name:30s}"
    for sel in (np.abs(z) > 0.3, np.abs(z) < 0.3):
        ok = (r.maps[1] != hp.UNSEEN) & sel
        e = (r.maps - truth.maps)[:, ok] * 1e6
        line += f" | {e[0].std():5.1f} {e[1].std():5.1f} {e[2].std():5.1f}"
    if r.gains:
        ge = np.array([g[t // 12000] / g0 - 1 for g, g0 in zip(r.gains, gains)])
        line += f" | {np.sqrt(np.mean(ge**2)):.1e} {ge.mean():+.1e}"
    if r.bandpass:
        line += " | " + " ".join(f"{v:+.2f}" for v in r.bandpass["fg_deriv1"].values())
    print(line)


print(f"{'':30s} | out of band I Q U [uK] | on the band I Q U    | gains rms, abs    | band shifts [GHz]")

                               | out of band I Q U [uK] | on the band I Q U    | gains rms, abs    | band shifts [GHz]


## A: no gain variation, no bandpass mismatch

In [2]:
report("A: maps", make_maps(ignore_gains(make(False, False)), nside, coordinates="E", params=params, sky_model=model))

A: maps                        |   8.8  12.4  12.6 |  10.5  14.8  14.6


## B: gain variation

In [3]:
B = make(True, False)
report("B: gains ignored", make_maps(ignore_gains(B), nside, coordinates="E", params=params, sky_model=model))
report("B: calibrated", make_maps(B, nside, coordinates="E", params=params, sky_model=model, calibration=cal))

B: gains ignored               |  55.0  18.7  17.3 |  51.9  25.1  29.0


B: calibrated                  |   9.1  12.9  13.0 |  11.1  15.3  15.0 | 3.2e-03 -1.6e-04


Ignoring the gains, a 1% error on the 3.4 mK dipole leaves tens of µK in I,
and the different gains of the two detectors of a pair leak I into Q and U.
Calibrated, the maps are back to the noise of A, plus a few percent from the
statistical error of the gains (3e-3 per 20-minute period).

## C: gain variation and bandpass mismatch

In [4]:
C = make(True, True)
report("C: both ignored", make_maps(ignore_gains(C), nside, coordinates="E", params=params, sky_model=model))
report("C: calibration only", make_maps(C, nside, coordinates="E", params=params, sky_model=model, calibration=cal))
report("C: calibration + bandpass", make_maps(C, nside, coordinates="E", params=params, sky_model=model, calibration=cal,
                                              bandpass_templates={"fg_deriv1": F1}))
print("true band shifts - their mean [GHz]:", shift - shift.mean())

C: both ignored                |  55.0  18.1  16.3 |  52.0  18.2  17.6


C: calibration only            |   9.2  13.6  13.4 |  11.1  20.5  19.8 | 6.6e-03 -1.7e-04


C: calibration + bandpass      |   9.1  12.9  13.0 |  11.1  15.4  15.0 | 3.2e-03 -1.3e-04 | +1.57 -1.07 +0.71 -1.21
true band shifts - their mean [GHz]: [ 1.5 -1.   0.8 -1.3]


With the calibration only, I is right but Q and U on the Galactic band are
worse than in A, and the gain errors double: the fit takes the different dust
seen by each detector for a gain error. With the bandpass template
($\partial F/\partial\nu$) the maps and the gains are as in B, and the band
shifts are found within ~0.1 GHz. (With both ignored, Q/U on the band happen
to be less wrong than in B: here the leakage of the bands partly cancels that
of the gains.)